# garimpo-uncertainty on Google Colab

This notebook runs the replication package of the dissertation end to end on Colab: it tests the
code, downloads the data, runs the pilot and the four full runs, and merges their tables. Each step
is a section below; the documents in `docs/` give the reasons behind every choice.

## How to use it

1. Do the one-time setup in **Before the first session**.
2. At the start of **every** session, run the three cells of **Session setup**, in order.
3. Then run **one** step, on the runtime its table row names.

> **Never use Runtime → Run all.** Each step ends by releasing the runtime, so the cells after it
> would run on no machine, or on the wrong one.

| Step | Runtime | How often | Keeps in Drive |
|---|---|---|---|
| 1. Test suite | GPU | Once, before the first paid run | nothing |
| 2. Download the data | CPU | Once | `data.tar.zst` |
| 3. Pilot | L4 GPU | Until every variant passes | `runs/pilot/`, `logs/` |
| 4. Full run | GPU | Once per variant, one per session | `runs/<variant>-f1.0/`, `logs/` |
| 5. Merge the results | CPU | Once, after the four full runs | `outputs/` |

Change the runtime in **Runtime → Change runtime type** before running a step's cells.

## Before the first session

The repository is private, so every session clones it with a read-only token.

1. On GitHub, create a fine-grained personal access token with access to this repository only and
   the permission **Contents: Read-only**.
2. In Colab, open the key icon (**Secrets**) on the left, add a secret named `GH_TOKEN` with the
   token as its value, and turn on **Notebook access**.
3. Open this notebook with **File → Upload notebook**, or from **File → Open notebook → GitHub**
   with **Include private repos** ticked.

The token is read-only on purpose: code changes are committed and pushed from your own machine,
never from a session, and the next session's clone picks them up (`docs/colab.md`).

## Session setup

Run these three cells at the start of every session. The runtime's disk is erased when the session
ends, so each session clones the code again and restores the data and the runs from Drive.

### 1. Settings and Google Drive

Mounts Drive, which asks for your permission, and defines the paths and helpers the steps use.
`DATA` must stay `/content/garimpo-data`: every run records the path of its data, and a run
restored under another path refuses to continue (`docs/stages.md`, run identity).

In [ ]:
# @title Settings and Google Drive
import contextlib
import os
import signal
import subprocess
from pathlib import Path

from google.colab import drive, runtime, userdata

drive.mount("/content/drive")

REPO = "github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone, erased with the session
DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions: data archive, runs, logs
DATA = "/content/garimpo-data"  # never change it: runs record this path
TOKEN = userdata.get("GH_TOKEN")

os.environ["GARIMPO_DATA_DIR"] = DATA
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]  # where uv installs itself

VERDICTS = ("PIPELINE CHECK: PASS", "PIPELINE CHECK: FAIL")


def print_verdicts(log_names: list[str]) -> None:
    """Print the last pipeline-check line of each job's log.

    A cell carries on after a failed shell line, so a job's outcome is this line, not the cell's
    status. It never raises, so the cell always goes on to release the GPU.
    """
    for name in log_names:
        log = Path("outputs/logs") / f"{name}.log"
        text = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
        found = [line for line in text.splitlines() if line.startswith(VERDICTS)]
        print(name, found[-1] if found else "NO VERDICT: the job or its check died; see the log")

### 2. Install the tools and the code

Installs `rsync` and `zstd`, which copy and unpack the data, and `uv`, which builds the Python
environment from the versioned `uv.lock`. `--recurse-submodules` also clones `compass/`, the
adapted COMPASS method.

In [ ]:
# @title Install the tools and the code
!apt-get -qq install -y rsync zstd > /dev/null
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone --recurse-submodules https://{TOKEN}@{REPO} {CODE}
%cd {CODE}
!uv sync
!git log -1 --format='Code at %h, %cs: %s'

### 3. Restore the data and the runs from Drive

Unpacks the data archive written by step 2 and copies back the runs of earlier sessions, so their
finished stages skip. Steps 1 and 5 do not read the data: untick `RESTORE_DATA` for them to save
the time of unpacking it. Colab keeps the box as you left it, so tick it again before steps 2 to 4. `tar` and `rsync -a` keep modification times, which the stages compare to
decide what to skip (`docs/colab.md`). Before step 2 there is no archive yet, and the cell says so.

The last line shows the free disk. At the 100% fraction the store takes 69.2 GB and `.venv` 6.9 GB,
plus the runs (`README.md`).

In [ ]:
# @title Restore the data and the runs from Drive
RESTORE_DATA = True  # @param {type:"boolean"}

!mkdir -p {DRIVE}/runs {DATA} runs outputs/logs
archive = Path(DRIVE) / "data.tar.zst"
if not RESTORE_DATA:
    print("Data not restored, as RESTORE_DATA asks.")
elif archive.exists():
    !tar -I zstd -xf {archive} -C {DATA}
else:
    print(f"No {archive} yet: run step 2 on a CPU runtime first. Steps 1 and 5 do not need it.")
!rsync -a {DRIVE}/runs/ runs/
!df -h /content

## Step 1. Test suite on a GPU

**Runtime: GPU. Once, before the first paid run.**

Runs the end-to-end suite on the GPU. It builds synthetic data of its own, so it needs neither
step 2 nor the archive. On your own machine and in CI the suite runs on the CPU; this run checks
that the same code also works on CUDA (`docs/testing.md`, "On a GPU").

Every test should pass. The artifacts of each test are in `tests/artifacts/`, which the file browser
on the left shows; the session's disk erases them, so read the failures before releasing the GPU.

In [ ]:
# @title Run the test suite on the GPU
!GARIMPO_TEST_DEVICE=cuda uv run --no-sync pytest -q

In [ ]:
# @title Release the GPU (after reading the failures, if any)
runtime.unassign()

## Step 2. Download the data

**Runtime: CPU. Once.**

Downloads everything the runs read, then packs it into one archive in Drive, which the session
setup unpacks every time:

1. `metadata`: the dataset's labels and the tile metadata.
2. `scenes` from the 1% list, then from the 100% list: the Sentinel-2 scenes, cut into cells of
   512 × 512 pixels. The 100% list contains the 1% list, so the second call downloads only the rest.
3. `weights`: the ImageNet and Sentinel-2 MoCo starting weights.
4. The archive `data.tar.zst`, written to Drive.

The archive is written only when every download command ends without an error, first to a
temporary file, so a failed or interrupted cell never replaces a good archive in Drive; Drive needs
room for both while the new one is written. The last line says which happened. A tile that fails
does not stop the command: each `scenes` call ends with `done: N tiles in ..., K not ok`, and the
reason of each failed tile is in its sidecar. Rerunning the cell retries only the tiles not ok,
after the session setup restores the finished ones. A tile whose scene does not exist
(`missing_item`) stays not ok; the pilot's check report counts those scenes. The `date` lines time each download. The time the 1% list takes tells you whether the 100% list
fits in one session; a session that ends mid-download loses everything on the local disk.
`--resampling-20m bilinear` is the decided resampling of the 20 m bands (`docs/decisions.md`).

In [ ]:
# @title Download the data, archive it in Drive, release the runtime
!date \
  && uv run --no-sync python scripts/download_data.py metadata && date \
  && uv run --no-sync python scripts/download_data.py scenes \
       --tiles outputs/tiles/tiles_1pct.json --resampling-20m bilinear && date \
  && uv run --no-sync python scripts/download_data.py scenes \
       --tiles outputs/tiles/tiles_100pct.json --resampling-20m bilinear && date \
  && uv run --no-sync python scripts/download_data.py weights \
       --encoder-weights imagenet sentinel2_moco \
  && tar --format=posix -I 'zstd -3 -T0' -cf {DRIVE}/data.tar.zst.tmp -C {DATA} . \
  && mv {DRIVE}/data.tar.zst.tmp {DRIVE}/data.tar.zst \
  && echo "DATA: PASS, archive written to Drive" \
  || (rm -f {DRIVE}/data.tar.zst.tmp; echo "DATA: FAIL, see the error above; Drive keeps its previous archive, if any")
runtime.unassign()

To download on your own machine instead and upload the archive to Drive as
`garimpo/data.tar.zst`, see `docs/full-run.md`, "Download the store".

## Step 3. Pilot

**Runtime: L4 GPU. Until every variant passes.**

The pilot runs the real pipeline at the 1% training fraction, to debug the implementation on a
GPU before the paid runs. Its numbers say nothing about any method. It runs the three U-Net
starting weights one after another: `none` (from scratch), `imagenet` and `sentinel2_moco`. Each
writes to `runs/pilot/unet-<weights>/` and its log to `outputs/logs/pilot-<weights>.log`; both are
copied to Drive when the cell ends.

Read the result in the lines the cell prints last, one per variant:

- `PIPELINE CHECK: PASS`: every stage finished and every check held.
- `PIPELINE CHECK: FAIL`: the log above that line names the failing stage and its error.
- `NO VERDICT`: the job or its check died before printing a verdict; the log says where.

The full report is explained in `docs/pilot.md`.

**Rerunning after a code fix.** Push the fix from your machine, start a new session, and set
`FORCE_FROM` to the first stage the fix affects. That stage and every later one run again; earlier
stages skip. The stages, in order: `train`, `infer`, `fit_directions`, `layer_monotonicity`,
`choose_layer`, `endpoint_monotonicity`, `compass_scores`, `calibrate`, `generate_intervals`,
`compute_metrics`. Set `FORCE_VALIDATE` to `1` to redo the data validation as well. `FORCE_FROM=train`
refuses while `last.pt` holds an unfinished training: leave it empty to resume that training.
Leave both empty on the first run (`docs/pilot.md`, "Rerun after a code fix").

In [ ]:
# @title Run the pilot, save it to Drive, release the GPU
FORCE_FROM = ""  # @param {type:"string"}
FORCE_VALIDATE = ""  # @param ["", "1"]
PILOT_WEIGHTS = ["none", "imagenet", "sentinel2_moco"]

for weights in PILOT_WEIGHTS:
    !FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={FORCE_VALIDATE} ENCODER_WEIGHTS={weights} \
      bash slurm/pilot.sbatch 2>&1 | tee outputs/logs/pilot-{weights}.log
!rsync -a runs/ {DRIVE}/runs/ && rsync -a outputs/logs/ {DRIVE}/logs/
print_verdicts([f"pilot-{weights}" for weights in PILOT_WEIGHTS])
runtime.unassign()

## Step 4. Full run

**Runtime: GPU. One variant per session, four in all.**

Start only when the pilot passes and every value in `docs/decisions.md` is fixed. The experiment
is four runs at the 100% training fraction, all on a ResNet-18 encoder (`docs/runs.md`):

| `ENV` | Model |
|---|---|
| `unet_scratch-f1.0` | U-Net from scratch |
| `unet_imagenet-f1.0` | U-Net from ImageNet weights |
| `unet_sentinel2-f1.0` | U-Net from Sentinel-2 MoCo weights |
| `convnet-f1.0` | smp's DeepLabV3 |

Each run reads its values from `slurm/<ENV>.env`. Write that file on your own machine, then commit
and push it, since the session's token cannot push:

```bash
cp slurm/full.env.example slurm/unet_scratch-f1.0.env   # fill in every value from docs/decisions.md
```

The run chooses the COMPASS-J layer from the validation role and records why in
`chosen_layer.json`. While it runs, a background copy saves `runs/` to Drive every 30 minutes, so a
session that ends mid-run keeps its finished stages and its last checkpoint. The cell stops it,
along with any copy in progress, before the final copy, so two copies never write to Drive at once. To resume, run the
session setup and this cell again with the same `ENV` and `FORCE_FROM` empty: training continues
from the checkpoint. `FORCE_FROM` and `FORCE_VALIDATE` work as in the pilot.

In [ ]:
# @title Run one variant, save it to Drive, release the GPU
ENV = "unet_scratch-f1.0"  # @param ["unet_scratch-f1.0", "unet_imagenet-f1.0", "unet_sentinel2-f1.0", "convnet-f1.0"]
FORCE_FROM = ""  # @param {type:"string"}
FORCE_VALIDATE = ""  # @param ["", "1"]

if Path(f"slurm/{ENV}.env").exists():
    backup_loop = f"while sleep 1800; do rsync -a --exclude '*.tmp' runs/ {DRIVE}/runs/; done"
    backup = subprocess.Popen(  # noqa: S603 -- runs the line above, which takes no outside input
        ["/bin/bash", "-c", backup_loop],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,  # its own process group, so one signal also stops a running rsync
    )
    !NUM_WORKERS={os.cpu_count()} FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={FORCE_VALIDATE} \
      bash slurm/full.sbatch slurm/{ENV}.env 2>&1 | tee outputs/logs/{ENV}.log
    with contextlib.suppress(ProcessLookupError):  # nothing before unassign() may raise
        os.killpg(backup.pid, signal.SIGTERM)
    backup.wait()
    !rsync -a runs/ {DRIVE}/runs/ && rsync -a outputs/logs/ {DRIVE}/logs/
    print_verdicts([ENV])
else:
    print(f"slurm/{ENV}.env is not in the repository: write it, push it, and start a new session.")
runtime.unassign()

## Step 5. Merge the results

**Runtime: CPU. Once, after the four full runs pass.**

The session setup has restored the four runs from Drive. This cell merges the tables of the four
runs of step 4, named one by one, into `outputs/` (`results.csv` and the comparison tables, listed
in `docs/outputs.md`). Only a successful merge is copied to Drive; the last line says whether the
merge and the copy succeeded. Commit those tables from your own machine: they are the numbers the
dissertation reports.

In [ ]:
# @title Merge the four runs' tables and copy them to Drive
!uv run --no-sync python scripts/merge_results.py --out outputs --runs \
    runs/unet_scratch-f1.0 runs/unet_imagenet-f1.0 runs/unet_sentinel2-f1.0 runs/convnet-f1.0 \
  && rsync -a outputs/ {DRIVE}/outputs/ \
  && echo "MERGE: PASS, tables copied to Drive" \
  || echo "MERGE: FAIL, see the error above"

In [ ]:
# @title Release the runtime
runtime.unassign()

## Where everything is

| Path | What | Lifetime |
|---|---|---|
| `MyDrive/garimpo/data.tar.zst` | The data store, packed by step 2 | Kept |
| `MyDrive/garimpo/runs/` | Every run: checkpoints, tables, stage statuses | Kept |
| `MyDrive/garimpo/logs/` | The log of every pilot and full-run job | Kept |
| `MyDrive/garimpo/outputs/` | The merged tables of step 5 | Kept |
| `/content/garimpo-uncertainty/` | The clone of the code | Erased with the session |
| `/content/garimpo-data/` | The unpacked store | Erased with the session |

## Further reading

| Document | Covers |
|---|---|
| `docs/colab.md` | Why the cells are written as they are |
| `docs/pilot.md` | The pilot's check report |
| `docs/full-run.md` | The full run, downloading on your own machine, rerunning one stage |
| `docs/decisions.md` | The values to fix before the full run |
| `docs/stages.md` | Every stage, its statuses, skips and refusals |
| `docs/failure-modes.md` | The ways the pipeline can fail and what catches each |